# CAMELS: Plot EMO1 Time Series for the Website
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 22-04-2026<br>

**Introduction:**<br>
This script creates plot to visualize the daily meteorological time series generated from the EMO-1 dataset.

In [1]:
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

from ocab.config import Config
from ocab.plots.meteo import plot_meteo_timeseries


## Configuration


In [2]:
cfg = Config('config_CAMELS_v200.yml')

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries' / 'meteo' / 'EMO1'
path_plots = path_in / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)

variables = {
    'ta_mean': 'temp_degC', 
    'pr_mean': 'precip_mm', 
    'e0_mean': 'pet_mm',
}

## Create plots

In [ ]:
# load stations
stations = gpd.read_file(cfg.path_gis / 'stations.geojson').set_index('id')

# process timeseries for each station
for ID in tqdm(stations.index, desc='stations'):

    # meteo timeseries
    try:
        meteo = pd.read_parquet(path_in / f'{ID:04d}.parquet').loc[ID]
        meteo.rename(columns=variables, inplace=True, errors='ignore')
        
        # correct dates
        meteo.index = meteo.index.date - pd.Timedelta(days=1) # EMO1 uses and end of time step convention
        meteo.index.name = 'date'
        meteo.index = pd.to_datetime(meteo.index)
    except Exception as e:
        print(f'Error loading meteo timeseries for station {ID:04d}: {e}')
        continue

    # extract attributes and time series
    attrs = stations.loc[ID]

    # create time series plot
    try:
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title(), 
            attrs['river'].title(), 
            attrs['basin'].title()
        )
        fig = plot_meteo_timeseries(meteo, title=title, save=True)
        fig.write_html(path_plots / f'{ID:04d}.html')
    except:
        print(f"The plot for time series {ID} couldn't be created")

stations:   0%|          | 0/1009 [00:00<?, ?it/s]